# 03 — Feature Engineering and Preprocessing

This notebook prepares the student data for later predictive modeling. It defines the regression target, keeps identifiers out of the model inputs, creates a train/test split, and builds a reusable preprocessing pipeline. **No predictive model is trained here.**

The preprocessing pipeline is fit using training rows only. That keeps information from the test set out of learned preprocessing choices and gives us a fairer evaluation in the later modeling module.

## 1. Imports and reproducibility

A fixed random seed makes the train/test split repeatable, which helps us compare later experiments consistently. The notebook locates the project root from either the root folder or the `notebooks/` folder, so the dataset path stays project-relative.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "Final_CGPA"
IDENTIFIER_COLUMN = "Student_ID"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "raw" / "Student_data.csv").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Student_data.csv"
assert DATA_PATH.is_file(), "Expected data/raw/Student_data.csv relative to the project root."

print(f"Dataset path: {DATA_PATH.relative_to(PROJECT_ROOT)}")

## 2. Load and inspect the data

We inspect the table before preparing it: its dimensions and data types, missing values, and exact duplicate rows. Missing values are handled in the preprocessing pipeline even if this dataset currently has none, so the same workflow remains usable if future data contains gaps.

In [ ]:
data = pd.read_csv(DATA_PATH)

print(f"Shape: {data.shape[0]:,} rows × {data.shape[1]} columns")
display(data.dtypes.rename("dtype").to_frame())
display(data.isna().sum().rename("missing_values").to_frame())
duplicate_count = int(data.duplicated().sum())
print(f"Exact duplicate rows: {duplicate_count}")

assert not data.empty, "The loaded dataset is empty."
assert TARGET_COLUMN in data.columns, f"Required target {TARGET_COLUMN!r} is missing."
assert IDENTIFIER_COLUMN in data.columns, f"Required identifier {IDENTIFIER_COLUMN!r} is missing."
assert pd.api.types.is_numeric_dtype(data[TARGET_COLUMN]), "The regression target must be numeric."
assert duplicate_count == 0, "Unexpected duplicate rows found; inspect them before training."
assert data[TARGET_COLUMN].notna().all(), "Target values cannot be missing for supervised training."

## 3. Define the target and candidate predictors

`Final_CGPA` is the continuous regression target. `Student_ID` is retained in the original table for identification and record keeping, but excluded from model features: an arbitrary identifier does not describe a student's academic circumstances and could encourage memorization.

The remaining columns are candidate predictors. `Previous_CGPA` is appropriate when it is recorded before the period whose final CGPA we want to predict. It is a strong historical measure, but its association is not proof of causation. If it were collected after the prediction time, it would not be a valid input.

In [ ]:
excluded_columns = [TARGET_COLUMN, IDENTIFIER_COLUMN]
feature_columns = [column for column in data.columns if column not in excluded_columns]
X = data.loc[:, feature_columns].copy()
y = data.loc[:, TARGET_COLUMN].copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Target: {TARGET_COLUMN}")
print(f"Excluded identifier: {IDENTIFIER_COLUMN}")
print(f"Candidate predictor columns ({len(feature_columns)}): {feature_columns}")
print(f"Numerical predictors ({len(numeric_features)}): {numeric_features}")
print(f"Categorical predictors ({len(categorical_features)}): {categorical_features}")

assert TARGET_COLUMN not in X.columns, "The target must not appear among the inputs."
assert IDENTIFIER_COLUMN not in X.columns, "The identifier must not appear among the inputs."
assert len(feature_columns) > 0, "No candidate predictors were selected."
assert set(numeric_features).isdisjoint(categorical_features), "Feature type groups overlap."
assert set(numeric_features) | set(categorical_features) == set(feature_columns), "Some predictors were not assigned a type."

## 4. Split before learning preprocessing

We reserve 20% of the rows as a test set and use a fixed random seed. The split happens **before** imputers, scaling, or category encoding are fit. This prevents preprocessing from learning statistics or category information from the test rows. The target remains separate from the input features throughout.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")

assert len(X_train) + len(X_test) == len(data), "The split lost or duplicated rows."
assert len(y_train) == len(X_train) and len(y_test) == len(X_test), "Input/target split sizes do not match."
assert set(X_train.index).isdisjoint(set(X_test.index)), "A row appears in both splits."

## 5. Build the preprocessing pipeline

Numerical columns use median imputation followed by standardization. The median is less affected by extreme values than the mean, and scaling puts numerical variables on comparable scales for models that are sensitive to feature magnitude. Categorical columns use most-frequent-value imputation followed by one-hot encoding; unknown categories at prediction time are safely ignored.

The `ColumnTransformer` applies the right steps to each column group. Wrapping it in a `Pipeline` makes the preprocessing easy to attach to a model later and ensures `fit` is called only on training data.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot_encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], remainder="drop")

preprocessing_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
])

## 6. Fit on training data and validate the transformed data

`fit_transform` learns imputation values, scaling statistics, and category levels from `X_train` only. We then call `transform` on `X_test`, reusing those training-learned steps without refitting. This is the key leakage-prevention step.

These transformed arrays are prepared inputs for later modeling. This notebook does not fit a predictor or claim that preprocessing improves prediction accuracy.

In [ ]:
X_train_transformed = preprocessing_pipeline.fit_transform(X_train)
X_test_transformed = preprocessing_pipeline.transform(X_test)
feature_names_out = preprocessing_pipeline.named_steps["preprocessor"].get_feature_names_out()

print(f"Transformed training shape: {X_train_transformed.shape}")
print(f"Transformed test shape: {X_test_transformed.shape}")
print(f"Number of transformed features: {len(feature_names_out)}")
print("First transformed feature names:", feature_names_out[:10].tolist())

assert X_train_transformed.shape[0] == len(X_train), "Training transform row count changed."
assert X_test_transformed.shape[0] == len(X_test), "Test transform row count changed."
assert X_train_transformed.shape[1] == X_test_transformed.shape[1], "Train and test feature counts differ."
assert len(feature_names_out) == X_train_transformed.shape[1], "Feature names do not match transformed columns."
assert all(IDENTIFIER_COLUMN not in name for name in feature_names_out), "Identifier leaked into transformed features."
assert np.isfinite(X_train_transformed).all(), "Training transform contains NaN or infinite values."
assert np.isfinite(X_test_transformed).all(), "Test transform contains NaN or infinite values."
print("All preprocessing assertions passed.")

## Summary and next step

- `Final_CGPA` is kept separately as the regression target.
- `Student_ID` is not used as a predictive feature.
- Numerical and categorical inputs follow separate preprocessing steps.
- The random train/test split happens before preprocessing is fit.
- The fitted preprocessing pipeline learns only from training inputs and can be attached to a regression estimator later.

The next module can compare regression and academic-risk classification models using this same split and preprocessing design. Model quality must be established with held-out evaluation; preprocessing alone does not establish predictive performance.